# TP Jour 2 — Modèle d'Exécution Spark, Jointures Distribuées & Couche Silver (Delta Lake)

### Objectifs de la séance
1. Distinguer transformations **Narrow** vs **Wide (Shuffle)** sur **Spark Web UI** (`http://localhost:4040`).
2. Nettoyer, typer et pré-agréger les tables de la couche **Bronze** sans créer de produit cartésien N-M.
3. Construire la table analytique **`/lakehouse/silver/orders`** en exploitant le **Broadcast Hash Join** (`F.broadcast`).
4. Pratiquer les garanties **ACID de Delta Lake** : `_delta_log`, **Schema Enforcement**, **Upsert (`MERGE INTO`)** et **Time Travel (`VERSION AS OF`)**.

> **Auto-validation :** Une fois terminé, testez votre notebook avec `task test:j2`.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from delta.tables import DeltaTable

spark = (
    SparkSession.builder
    .appName("J2_Spark_Delta_Silver")
    .getOrCreate()
)

orders_bz = spark.read.format("delta").load("/lakehouse/bronze/orders")
items_bz = spark.read.format("delta").load("/lakehouse/bronze/order_items")
payments_bz = spark.read.format("delta").load("/lakehouse/bronze/order_payments")
reviews_bz = spark.read.format("delta").load("/lakehouse/bronze/order_reviews")
customers_bz = spark.read.format("delta").load("/lakehouse/bronze/customers")
products_bz = spark.read.format("delta").load("/lakehouse/bronze/products")
cat_trans_bz = spark.read.format("delta").load("/lakehouse/bronze/category_translation")

## 1. Diagnostic Qualité des Données

### Exercice 1.1 — Compter les valeurs nulles par colonne sur `orders_bz`
Affichez le nombre de valeurs `NULL` sur les colonnes clés de `orders_bz` (`order_id`, `customer_id`, `order_status`, `order_purchase_timestamp`, `order_delivered_customer_date`, `order_estimated_delivery_date`).

In [ ]:
cols_to_check = [
    "order_id", "customer_id", "order_status",
    "order_purchase_timestamp", "order_delivered_customer_date", "order_estimated_delivery_date",
]

# TODO : Calculer et afficher le nombre de valeurs nulles pour chaque colonne de `cols_to_check`
null_counts = None
assert null_counts is not None, "TODO : Calculez le DataFrame `null_counts`."
null_counts.show(truncate=False)

## 2. Nettoyage, Broadcast Join & Prévention du Produit Cartésien N-M

> **Piège classique N-à-M :** Une commande (`order_id`) peut avoir plusieurs lignes dans `order_items` ET plusieurs séquences dans `order_payments`. Les joindre directement avant agrégation multiplie artificiellement les montants !

### Exercice 2.1 — Enrichir `products` par Broadcast Join et créer `/lakehouse/silver/order_items`
1. Dédupliquez `products_bz` sur `product_id` et effectuez un **`left` join diffusé (`F.broadcast(...)`)** avec `cat_trans_bz` sur `product_category_name`. Créez une colonne `category` via `F.coalesce(product_category_name_english, product_category_name, F.lit("unknown"))`.
2. Inspectez `products_clean.explain("simple")` pour vérifier la présence de `BroadcastHashJoin`.
3. Filtrez `items_bz` (`price > 0` et `freight_value >= 0`), dédupliquez sur `["order_id", "order_item_id"]`, joignez `products_clean` en broadcast, ajoutez `line_total = round(price + freight_value, 2)` et sauvegardez dans **`/lakehouse/silver/order_items`**.

In [ ]:
# TODO 1 : Construire `products_clean` avec F.broadcast(cat_trans_bz)
products_clean = None
assert products_clean is not None, "TODO : Construisez `products_clean` avec un Broadcast Join."
products_clean.explain("simple")

# TODO 2 : Construire `silver_items` et l'écrire en Delta dans `/lakehouse/silver/order_items`
silver_items = None
assert silver_items is not None, "TODO : Construisez `silver_items` et sauvegardez `/lakehouse/silver/order_items`."
...

### Exercice 2.2 — Pré-agréger à la maille `order_id`
Construisez 3 DataFrames agrégés par `order_id` :
- `items_per_order` : `total_items` (`count`), `items_amount` (`sum(price)`), `freight_amount` (`sum(freight_value)`), `total_amount` (`sum(line_total)`), `category` (`first("category", ignorenulls=True)`).
- `payments_per_order` : `paid_amount` (`sum(payment_value)`), `max_installments` (`max(payment_installments)`), `primary_payment_type` (`first("payment_type", ignorenulls=True)`).
- `reviews_per_order` (notes entre 1 et 5) : `review_score` (`round(avg(review_score), 2)`).

In [ ]:
# TODO : Agréger items_per_order, payments_per_order et reviews_per_order par `order_id`
items_per_order = None
payments_per_order = None
reviews_per_order = None

customers_clean = (
    customers_bz
    .dropDuplicates(["customer_id"])
    .select("customer_id", "customer_unique_id", "customer_city", "customer_state")
)

assert all(df is not None for df in [items_per_order, payments_per_order, reviews_per_order]), (
    "TODO : Complétez les 3 pré-agrégations par `order_id`."
)

## 3. Assemblage de la Table Maîtresse `/lakehouse/silver/orders`

### Exercice 3.1 — Calculer les délais logistiques et sauvegarder `/lakehouse/silver/orders`
À partir de `orders_bz` dédupliqué sur `order_id` :
1. Convertissez `order_purchase_timestamp`, `order_approved_at`, `order_delivered_customer_date` et `order_estimated_delivery_date` avec `F.to_timestamp(...)`.
2. Ajoutez :
   - `order_month` : `F.date_format("order_purchase_timestamp", "yyyy-MM")`
   - `delivery_days` : délai réel `(unix_timestamp(delivered) - unix_timestamp(purchase)) / 86400.0` arrondi à 2 décimales
   - `delay_days` : écart `(unix_timestamp(delivered) - unix_timestamp(estimated)) / 86400.0` arrondi à 2 décimales
   - `is_late` : `1` si `delay_days > 0`, sinon `0`
3. Filtrez les éventuelles anomalies où `delivery_days < 0` (conservez les `NULL` des commandes non encore livrées !).
4. Joignez `customers_clean` (`left`), `items_per_order` (`inner`), `payments_per_order` (`left`) et `reviews_per_order` (`left`), puis sauvegardez au format **Delta** dans **`/lakehouse/silver/orders`**.

In [ ]:
# TODO : Assembler `silver_orders` et l'écrire en Delta dans `/lakehouse/silver/orders`
silver_orders = None
assert silver_orders is not None, "TODO : Construisez `silver_orders`."

# TODO : Sauvegarder en Delta Lake dans `/lakehouse/silver/orders`
...

required_cols = {
    "order_id", "customer_unique_id", "customer_state", "order_status",
    "order_month", "category", "total_amount", "delivery_days", "delay_days", "is_late", "review_score",
}
actual_cols = set(spark.read.format("delta").load("/lakehouse/silver/orders").columns)
missing = required_cols - actual_cols
assert not missing, f"Colonnes manquantes dans /lakehouse/silver/orders : {missing}"
print("Table /lakehouse/silver/orders validée !")

## 4. Garanties ACID Delta Lake : `_delta_log`, `Schema Enforcement`, `MERGE INTO` & `Time Travel`

### Exercice 4.1 — Inspecter le commit JSON dans `_delta_log` et tester le `Schema Enforcement`

In [ ]:
# Lecture directe du premier fichier de transaction JSON Delta sur HDFS
spark.read.text("/lakehouse/silver/orders/_delta_log/00000000000000000000.json").show(8, truncate=110)

# Test de Schema Enforcement : écriture volontairement invalide rejetée par Delta Lake
bad_df = spark.createDataFrame([("ord_fake_1", "colonne_intruse")], ["order_id", "intruder_col"])
try:
    bad_df.write.format("delta").mode("append").save("/lakehouse/silver/orders")
except Exception as e:
    print("Schema Enforcement actif :", str(e).splitlines()[0])

### Exercice 4.2 — Upsert avec `DeltaTable.merge(...)` et audit par Time Travel (`versionAsOf`)
1. Prélevez 5 commandes au statut `"shipped"` dans `/lakehouse/silver/orders` et préparez un DataFrame de mise à jour passant leur statut à `"delivered"`, `delivery_days = 12.5` et `is_late = 0`.
2. Exécutez un `MERGE INTO` via `DeltaTable.forPath(spark, "/lakehouse/silver/orders")`.
3. Affichez `DESCRIBE HISTORY delta.'/lakehouse/silver/orders'` et comparez le nombre de commandes `"shipped"` entre `versionAsOf = 0` et la version courante !

In [ ]:
# TODO : Réaliser le MERGE INTO sur 5 commandes 'shipped' puis comparer VERSION 0 vs VERSION COURANTE
target_delta = DeltaTable.forPath(spark, "/lakehouse/silver/orders")
...

spark.sql("DESCRIBE HISTORY delta.`/lakehouse/silver/orders`").select("version", "timestamp", "operation").show(truncate=False)